# Detector de layout (Diamond) — Kaggle
Treina um detector YOLO nas 9 classes do Diamond, com o dataset `detector_diamond_yolo`.
A T4 (16 GB) permite um modelo maior que o baseline local (4050, 6 GB).

**Sessao:** Accelerator = **GPU T4**, Internet = **On**. Add Input = dataset `detector-diamond-yolo`.
No fim, baixe `best.pt` e `results.csv` na aba *Output*; a avaliacao contra o ouro roda no projeto.

In [ ]:
import glob, os
cands = glob.glob('/kaggle/input/**/images/train', recursive=True)
assert cands, 'Dataset nao encontrado: Add Input -> detector-diamond-yolo'
RAIZ = os.path.dirname(os.path.dirname(cands[0]))
print('dataset em', RAIZ)
!pip install -q ultralytics

In [ ]:
import yaml
CLASSES = ['titulo','texto','lista','tabela','assinatura','cabecalho','rodape','figura','outro']
data = {'path': RAIZ, 'train': 'images/train', 'val': 'images/val',
        'nc': len(CLASSES), 'names': {i: c for i, c in enumerate(CLASSES)}}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.safe_dump(data, f, allow_unicode=True)
print(open('/kaggle/working/data.yaml').read())

In [ ]:
# Treino. yolov8m e um bom equilibrio na T4; troque por yolov8s (mais leve) ou yolov8l (mais forte).
from ultralytics import YOLO
modelo = YOLO('yolov8m.pt')
modelo.train(data='/kaggle/working/data.yaml', epochs=200, imgsz=1024, batch=16,
             device=0, project='/kaggle/working/runs', name='diamond', seed=42, patience=40)

In [ ]:
# Reune os arquivos para a aba Output
import glob, os, shutil
d = '/kaggle/working/runs/diamond'
for f in glob.glob(d+'/weights/*.pt') + glob.glob(d+'/*.csv') + glob.glob(d+'/*.png'):
    shutil.copy(f, '/kaggle/working/' + os.path.basename(f))
print('Pronto. Baixe best.pt e results.csv na aba Output.')
print(sorted(os.listdir('/kaggle/working')))